In [14]:
library(tidyverse)
library(vroom)
library(data.table)
library(pheatmap)
library(preprocessCore)
library(purrr)
library(RColorBrewer)
library(brms)

final_psi_table_filtered <- fread("/mnt/dawnccle2/processed_data/reprocess_250221/count_normalized_v4_merged/WT_all_samples_PSI_count_table.csv")
final_psi_table_filtered <- final_psi_table_filtered %>% 
  filter(!(condition %in% c("K562WT", "K562K700E"))) %>% 
  filter(!(condition %in% c("JHOM1", "RVH421", "KNS60", "OVTOKO"))) %>% 
  mutate(total_count = included_count + skipped_count) %>%
  filter(total_count >= 30) %>%
  mutate(index_offset = paste(index, offset, sep = "__")) %>% 
  separate(offset, into = c("upstream_offset", "downstream_offset", "const_offset"), sep = ":") %>% 
  mutate(upstream_offset = as.integer(upstream_offset)) %>% 
  mutate(downstream_offset = as.integer(downstream_offset)) %>%
  mutate(const_offset = as.integer(const_offset)) %>% 
  filter(abs(upstream_offset) != 1 & abs(downstream_offset)!= 1) %>% 
  dplyr::select(-upstream_offset, -downstream_offset, -const_offset) %>% 
  dplyr::select(-index, -mode) %>%
  mutate(PSI = included_count/(included_count + skipped_count))
# Filter out the sequences that are in the blacklist.
blacklist_sequences <- read.csv("/mnt/dawnccle2/melange/process_fastq_250221/03_convert_to_PSI/WT_all_samples_blacklist_sequences.csv")
final_psi_table_filtered <- final_psi_table_filtered %>% 
  filter(!(index_offset %in% blacklist_sequences$index_offset))


dt <- as.data.table(final_psi_table_filtered)

# Compute PSI by condition efficiently
psi_by_condition <- dt[, .(PSI = mean(PSI, na.rm = TRUE), num_rep = .N), by = .(condition, index_offset)][
  num_rep >= 2, .(condition, index_offset, PSI)]  # Filter out groups with <2 replicates

psi_table_pivot <- psi_by_condition %>%
  select(condition, index_offset, PSI) %>%
  pivot_wider(names_from = c(condition), values_from = PSI)

psi_table_pivot_sample <- final_psi_table_filtered %>%
  select(sample, index_offset, PSI) %>%
  pivot_wider(names_from = c(sample), values_from = PSI) 

# Convert to matrix.
psi_table_mat <- as.matrix(psi_table_pivot_sample %>% select(-index_offset))
rownames(psi_table_mat) <- psi_table_pivot_sample$index_offset

In [15]:
# Filter out rows in psi_table_mat that's >20% NA.
psi_table_mat <- psi_table_mat[rowMeans(is.na(psi_table_mat)) < 0.2, ]

# Imput the missing values with the mean of the non-missing values per row. 
# Create a function to impute missing values with row means
impute_row_mean <- function(mat) {
  for (i in 1:nrow(mat)) {
    row_mean <- mean(mat[i, ], na.rm = TRUE)
    mat[i, is.na(mat[i, ])] <- row_mean
  }
  return(mat)
}

# Apply the imputation function
psi_table_mat <- impute_row_mean(psi_table_mat)

In [ ]:
# # Number of sequences to sample
# n_sequences <- 100

# # Sample n_sequences index_offsets (sequences) before pivoting
# set.seed(42)
# selected_indices <- sample(rownames(psi_table_mat), n_sequences)

# psi_subset <- psi_table_mat[selected_indices, ]

# # Reshape to long format
# psi_long <- as.data.frame(psi_subset) %>%
#   rownames_to_column("index_offset") %>%
#   pivot_longer(-index_offset, names_to = "sample", values_to = "psi") %>%
#   drop_na(psi) %>%
#   mutate(
#     index_idx = as.integer(factor(index_offset)),
#     sample_idx = as.integer(factor(sample))
#   )

# # Define priors
# priors <- c(
#   prior(normal(0, 0.05), class = "Intercept"),
#   prior(exponential(1), class = "phi"),               # beta precision
#   prior(exponential(1), class = "sd", group = "sample_idx"),
#   prior(exponential(1), class = "sd", group = "index_idx")
# )

# # Model formula:
# # - (1 | index_idx): varying intercept per sequence
# # - (1 | sample_idx): varying intercept per cell type (accounts for batch)
# fit <- brm(
#   formula = psi ~ (1 | index_idx) + (1 | sample_idx),
#   data = psi_long,
#   family = zero_one_inflated_beta(),
#   prior = priors,
#   cores = 4,
#   chains = 4,
#   iter = 4000,
#   control = list(adapt_delta = 0.99, max_treedepth = 15),
#   seed = 42
# )

# # Model summary
# print(summary(fit), digits = 3)

In [46]:
# Load libraries
library(cmdstanr)
library(tidyverse)

# Save Stan code to file
stan_model_code <- "
data {
  int<lower=1> N;
  int<lower=1> G;
  int<lower=1> B;
  array[N] int<lower=1, upper=G> seq_id;
  array[N] int<lower=1, upper=B> batch_id;
  vector<lower=0,upper=1>[N] psi;
}
parameters {
  vector[G] alpha_raw;
  real<lower=0> sigma_alpha;
  vector[B] batch_effect;
  array[G] real<lower=0> phi;
  real<lower=0, upper=1> p0;
  real<lower=0, upper=1> p1;
}
transformed parameters {
  vector[N] mu;
  for (n in 1:N) {
    // Constrain mu to be between 0 and 1 to avoid log1m errors
    mu[n] = inv_logit(logit(0.5) + alpha_raw[seq_id[n]] * sigma_alpha + batch_effect[batch_id[n]]);
  }
}
model {
  alpha_raw ~ normal(0, 1);
  sigma_alpha ~ normal(0, 0.05);
  batch_effect ~ normal(0, 1);
  phi ~ exponential(1);
  p0 ~ beta(1, 1);
  p1 ~ beta(1, 1);
  
  // Ensure p0 + p1 <= 1 to avoid probability errors
  if (p0 + p1 > 1) {
    reject(\"p0 + p1 must be <= 1\");
  }
  
  for (n in 1:N) {
    if (psi[n] == 0)
      target += log(p0);
    else if (psi[n] == 1)
      target += log(p1);
    else
      target += log1m(p0 + p1) +
                beta_lpdf(psi[n] | mu[n] * phi[seq_id[n]], (1 - mu[n]) * phi[seq_id[n]]);
  }
}
"

# Write model to file
stan_file <- tempfile(fileext = ".stan")
writeLines(stan_model_code, stan_file)

# Subsample 100 sequences
# n_sequences <- 1000
# set.seed(42)
# selected_sequences <- sample(rownames(psi_table_mat), n_sequences)
psi_subset <- psi_table_mat

# Reshape to long format
psi_long <- as.data.frame(psi_subset) %>%
  rownames_to_column("index_offset") %>%
  pivot_longer(-index_offset, names_to = "sample", values_to = "psi") %>%
  drop_na(psi) %>%
  mutate(
    batch_idx = as.integer(factor(sample)),
    seq_idx = as.integer(factor(index_offset))
  )

# Ensure values are strictly within [0, 1] to avoid boundary issues
psi_long$psi <- pmin(pmax(psi_long$psi, 0.00001), 0.99999)

# Prepare Stan data
stan_data <- list(
  N = nrow(psi_long),
  psi = psi_long$psi,
  G = length(unique(psi_long$seq_idx)),
  B = length(unique(psi_long$batch_idx)),
  seq_id = psi_long$seq_idx,
  batch_id = psi_long$batch_idx
)

# Compile model
mod <- cmdstan_model(stan_file)

# Try with more conservative settings for variational inference
fit_vb <- mod$variational(
  data = stan_data,
  seed = 42,
  output_samples = 1000,
  iter = 50000,  # Increase iterations
  grad_samples = 5,  # More gradient samples
  elbo_samples = 100,  # More ELBO samples
  adapt_engaged = TRUE,
  tol_rel_obj = 0.001  # Less strict convergence
)

# Check if fit was successful
if (fit_vb$return_codes() == 0) {
  # Extract posterior draws
  draws <- fit_vb$draws(format = "draws_matrix")
  
  # Get sequence-level effects only (exclude batch)
  alpha_idx <- grep("^alpha_raw\\[", colnames(draws))
  alpha_median <- apply(draws[, alpha_idx], 2, median)
  sigma_alpha_median <- median(draws[, "sigma_alpha"])
  
  # Reconstruct batch-corrected mu
  psi_long$psi_corrected <- plogis(
    qlogis(0.5) + alpha_median[psi_long$seq_idx] * sigma_alpha_median
  )
} else {
  # If variational inference fails, use semi-parametric quantile matching
  message("Variational inference failed. Using a simpler normalization approach.")
  
  # Step 1: Define pseudo-quantiles (ignore exact 0 and 1)
  psi_continuous <- psi_long %>%
    filter(psi > 0 & psi < 1)
  
  # Step 2: Center PSI by batch for middle values only
  batch_medians <- psi_continuous %>%
    group_by(batch_idx) %>%
    summarize(batch_median = median(psi, na.rm = TRUE))
  
  global_median <- median(psi_continuous$psi, na.rm = TRUE)
  
  # Step 3: Join and shift continuous PSI values
  psi_long <- left_join(psi_long, batch_medians, by = "batch_idx") %>%
    mutate(batch_median = ifelse(is.na(batch_median), global_median, batch_median))
  
  psi_long <- psi_long %>%
    mutate(psi_corrected = case_when(
      psi == 0 ~ 0,
      psi == 1 ~ 1,
      TRUE     ~ pmin(pmax(psi - (batch_median - global_median), 0), 1)
    ))
}

# Create a mapping from indices to original values
seq_idx_to_offset <- psi_long %>%
  select(seq_idx, index_offset) %>%
  distinct()

batch_idx_to_sample <- psi_long %>%
  select(batch_idx, sample) %>%
  distinct()

# Merge the mappings back to get original identifiers
psi_long_with_originals <- psi_long %>%
  left_join(seq_idx_to_offset, by = "seq_idx") %>%
  left_join(batch_idx_to_sample, by = "batch_idx")

# Ensure we have the original identifiers alongside the indices
psi_long <- psi_long_with_originals

------------------------------------------------------------ 
EXPERIMENTAL ALGORITHM: 
  This procedure has not been thoroughly tested and may be unstable 
  or buggy. The interface is subject to change. 
------------------------------------------------------------ 
Rejecting initial value: 
  Error evaluating the log probability at the initial value. 
Exception: p0 + p1 must be <= 1 (in '/tmp/RtmpozmvBX/model-116cc911edc39e.stan', line 35, column 4 to column 35) 
Gradient evaluation took 2.27135 seconds 
1000 transitions using 10 leapfrog steps per transition would take 22713.5 seconds. 
Adjust your expectations accordingly! 
Begin eta adaptation. 


Chain 1 stan::variational::advi::adapt_eta: Cannot compute ELBO using the initial variational distribution. Your model may be either severely ill-conditioned or misspecified.

Warning message:
“Fitting finished unexpectedly! Use the $output() method for more information.
”


Finished in  42.2 seconds.


Variational inference failed. Using a simpler normalization approach.



In [ ]:
# library(pheatmap)

# # Create heatmap of initial PSI values
# psi_long_mat_initial <- psi_long %>%
#   dplyr::select(batch_idx, seq_idx, psi) %>%
#   pivot_wider(names_from = batch_idx, values_from = psi) %>%
#   column_to_rownames("seq_idx")

# psi_long_mat_initial <- as.matrix(psi_long_mat_initial)

# # Create heatmap of initial PSI values 
# pheatmap(psi_long_mat_initial,
#         cluster_rows = TRUE,
#         cluster_cols = FALSE,
#         show_colnames = FALSE,
#         show_rownames = FALSE,
#         main = "Initial PSI Values"
# )




In [38]:
psi_long

index_offset,sample,psi,batch_idx,seq_idx,psi_corrected
<chr>,<chr>,<dbl>,<int>,<int>,<dbl>
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,769P-rep1,0.20553819,1,58,0.1893281
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,769P-rep2,0.20553819,2,58,0.1893281
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,769P-rep3,0.20553819,3,58,0.1893281
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,786O-rep1,0.35000000,4,58,0.1893281
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,786O-rep2,0.22580645,5,58,0.1893281
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,786O-rep3,0.18918919,6,58,0.1893281
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,8MGBA-rep1,0.27058824,7,58,0.1893281
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,8MGBA-rep2,0.26050420,8,58,0.1893281
ENSG00000143036.17;SLC44A3;chr1-94825832-94825942-94824493-94824635-94827506-94827643__0:0:0,8MGBA-rep3,0.18320611,9,58,0.1893281
